# Notebook overview: R8_7_CO_poilevel_changein_connections.ipynb

## What this notebook does
Aggregates user-user co-location ties to the POI level and estimates POI-specific post-disaster changes relative to counterfactual baselines. It computes POI-level interaction counts/residuals and runs POI-level regression/exploratory analyses.

## Inputs used
- Daily/monthly co-location JSONs and monthly graph outputs
- Observed and counterfactual interaction tables
- Cleaned POI table: CO_pois_TP.csv
- POI CBG/census covariates and POI fire-distance variables
- Python helper: R_monthly_poi_aggregation.py

## Outputs created
- POI-level interaction aggregation tables such as poi_agg_interac_*_{revision}.csv
- POI structural random/counterfactual merged tables
- POI-level z-score/residual tables
- Regression summaries and POI change plots/maps

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
pip install statsmodels 

In [ ]:
pip install seaborn

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
print("Current working directory:", os.getcwd())

In [ ]:
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
week_dir = os.path.join(graph_poi_dir, "user_connections" )
os.makedirs(week_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
import seaborn as sns

# Recerate merged weighted user - user edge at poi json

In [ ]:
WEEK_DATE_MAP = {
    # ---------------- POST DISASTER ----------------
    ("post_disaster", "Jan_2022"): ( "2022-01-06", "2022-02-02" ),
    ("post_disaster", "Feb_2022"): ( "2022-02-03", "2022-03-02" ),

    # ---------------- PRE DISASTER  ----------------
    ("pre_disaster", "Oct_2021"): ("2021-10-01", "2021-10-28"),
    ("pre_disaster", "Nov_2021"): ("2021-11-01", "2021-11-28"),
}

In [ ]:
import importlib
import R_monthly_poi_aggregation

importlib.reload(R_monthly_poi_aggregation)

from R_monthly_poi_aggregation import build_monthly_user_user_edges

In [ ]:
def run_all_monthly_aggregations(
    WEEK_DATE_MAP,
    stops_dir,
    revision="R11_2"  # or R8.1 / R11 etc
):

    for (phase, week), (window_start, window_end) in WEEK_DATE_MAP.items():

        # Prefix mapping
        if phase == "pre_disaster":
            prefix = "PDM"
        elif phase == "post_disaster":
            prefix = "PtDM"
        else:
            raise ValueError(f"Unknown phase: {phase}")

        # Month tag like Oct2021, Jan2022 etc
        month_tag = week.replace("_", "")

        print("\n" + "="*60)
        print(f"🚀 Processing: {phase} | {week}")
        print(f"📅 Window: {window_start} → {window_end}")
        print("="*60)

        build_monthly_user_user_edges(
            window_start=window_start,
            window_end=window_end,
            week_list=[week],   # single week per month
            stops_dir=stops_dir,
            phase=phase,
            revision=revision,
            prefix=prefix,
            month_tag=month_tag
        )

    print("\n🎯 All monthly aggregations completed.")

In [ ]:
run_all_monthly_aggregations(
    WEEK_DATE_MAP=WEEK_DATE_MAP,
    stops_dir=stops_dir,
    revision="R11_2"   
)

# Check basic counts for reference

In [ ]:
def inspect_monthly_jsons(stops_dir, revision):

    month_configs = [
        ("pre_disaster",  "PDM_Oct2021"),
        ("pre_disaster",  "PDM_Nov2021"),
        ("post_disaster", "PtDM_Jan2022"),
        ("post_disaster", "PtDM_Feb2022"),
    ]

    # Store users + dyads per phase
    phase_users = defaultdict(set)
    phase_dyads = defaultdict(set)

    for phase, folder in month_configs:

        file_path = os.path.join(
            stops_dir,
            f"{phase}/{folder}/{folder}_POI_weighted_colocation_{revision}.json"
        )

        print("\n" + "="*80)
        print(f"📂 {phase} | {folder}")
        print("📁 Path:", file_path)

        if not os.path.exists(file_path):
            print("❌ File does NOT exist.")
            continue

        print("✅ File exists")

        dyad_count = 0
        sample_entries = []
        users_in_month = set()
        dyads_in_month = set()

        with open(file_path, "r") as f:
            for i, line in enumerate(f):
                record = json.loads(line)
                dyad_count += 1

                dyad_key = list(record.keys())[0]

                # Convert "(u, v)" string back to tuple
                try:
                    u, v = ast.literal_eval(dyad_key)
                except:
                    continue

                users_in_month.update([u, v])
                dyads_in_month.add(tuple(sorted((u, v))))

                if i < 3:
                    sample_entries.append(record)

        phase_users[phase].update(users_in_month)
        phase_dyads[phase].update(dyads_in_month)

        print("🔗 Total dyads:", dyad_count)
        print("👥 Total unique users:", len(users_in_month))

        print("🔎 Sample entries (first 3):")
        for entry in sample_entries:
            print(entry)

        # Structure check
        if sample_entries:
            first_key = list(sample_entries[0].keys())[0]
            first_val = sample_entries[0][first_key]

            print("\n📌 Structure check:")
            print("Dyad key example:", first_key)
            print("Fields:", list(first_val.keys()))

            required_fields = [
                "weight",
                "pois",
                "poi_counts",
                "poi_overlap_weight",
                "composite_weight"
            ]

            missing = [f for f in required_fields if f not in first_val]

            if missing:
                print("⚠️ Missing fields:", missing)
            else:
                print("✅ Structure looks correct")

    # =====================================================
    #  Cross-phase overlap analysis
    # =====================================================

    print("\n" + "="*80)
    print("🔄 CROSS-PHASE ANALYSIS")
    print("="*80)

    pre_users  = phase_users["pre_disaster"]
    post_users = phase_users["post_disaster"]

    overlapping_users = pre_users.intersection(post_users)

    print("👥 Users in PRE:", len(pre_users))
    print("👥 Users in POST:", len(post_users))
    print("🔁 Users in BOTH (PRE ∩ POST):", len(overlapping_users))

    pre_dyads  = phase_dyads["pre_disaster"]
    post_dyads = phase_dyads["post_disaster"]

    print("\n🔗 PRE dyads total:", len(pre_dyads))
    print("🔗 POST dyads total:", len(post_dyads))

    # Dyads restricted to overlapping users
    pre_overlap_dyads = [
        (u, v)
        for (u, v) in pre_dyads
        if u in overlapping_users and v in overlapping_users
    ]

    post_overlap_dyads = [
        (u, v)
        for (u, v) in post_dyads
        if u in overlapping_users and v in overlapping_users
    ]

    print("\n🔗 PRE dyads where both users are in POST:", len(pre_overlap_dyads))
    print("🔗 POST dyads where both users are in PRE:", len(post_overlap_dyads))

#     # Optional but VERY useful:
#     # Dyads that survive from PRE to POST
#     surviving_dyads = set(pre_overlap_dyads).intersection(set(post_overlap_dyads))

#     print("\n🧬 Dyads present in BOTH PRE and POST:", len(surviving_dyads))

In [ ]:
inspect_monthly_jsons(stops_dir, revision="R7")

# Observed : Create combined df using all four json's to see change in co-location weights across time

In [ ]:
prephase = "pre_disaster"
preprefix = "PDM"
postphase = "post_disaster"
postprefix = "PtDM"
revision = "R11"

month_files = {
    "Oct2021": os.path.join(stops_dir, prephase, "PDM_Oct2021", f"{preprefix}_Oct2021_POI_weighted_colocation_{revision}.json"),
    "Nov2021": os.path.join(stops_dir, prephase, "PDM_Nov2021", f"{preprefix}_Nov2021_POI_weighted_colocation_{revision}.json"),
    "Jan2022": os.path.join(stops_dir, postphase, "PtDM_Jan2022", f"{postprefix}_Jan2022_POI_weighted_colocation_{revision}.json"),
    "Feb2022": os.path.join(stops_dir, postphase, "PtDM_Feb2022", f"{postprefix}_Feb2022_POI_weighted_colocation_{revision}.json"),
}

for m, p in month_files.items():
    print(m, "exists?" , os.path.exists(p), "|", p)

In [ ]:
def extract_users_from_month_file(json_path):
    users = set()

    with open(json_path, "r") as f:
        for line in f:
            if not line.strip():
                continue

            obj = json.loads(line)

            for k_str in obj.keys():
                u, v = parse_dyad_key(k_str)
                users.add(u)
                users.add(v)

    return users

In [ ]:
def parse_dyad_key(k_str: str):
    """
    Parse "(u, v)" -> (u, v) as strings, canonical ordered.
    """
    a, b = k_str.strip().strip("()").split(",")
    a, b = a.strip(), b.strip()
    # canonical ordering
    if a <= b:
        return a, b
    return b, a

def load_month_jsonl_to_long(jsonl_path: str, month_label: str) -> pd.DataFrame:
    """
    Returns long df: user_i, user_j, poi, month, weight
    where weight = poi_counts[poi] (POI-level dyad count for that month)

    Assumes each line is a dict with 1 entry: { "(u,v)": {...} }
    """
    rows = []
    with open(jsonl_path, "r") as f:
        for line in f:
            if not line.strip():
                continue

            obj = json.loads(line)

            for k_str, d in obj.items():
                ui, uj = parse_dyad_key(k_str)

                # Drop self-loops if present
                if ui == uj:
                    continue

                poi_counts = d.get("poi_counts", None)

                # If your file is the older one without poi_counts, stop loudly:
                if poi_counts is None:
                    raise KeyError(
                        f"{jsonl_path} is missing 'poi_counts'. "
                        f"Make sure you're reading the *_POI_weighted_colocation_*.json file."
                    )

                for poi, w in poi_counts.items():
                    rows.append({
                        "user_i": ui,
                        "user_j": uj,
                        "poi": poi,
                        "month": month_label,
                        "weight": int(w)
                    })

    return pd.DataFrame(rows)

In [ ]:
pre_months  = {"Oct2021", "Nov2021"}
post_months = {"Jan2022", "Feb2022"}

In [ ]:
pre_users = set()
post_users = set()

for month, path in month_files.items():
    if not os.path.exists(path):
        print(f"⚠️ Missing {month}, skipping")
        continue

    print(f"🔍 Scanning users in {month}")

    users_m = extract_users_from_month_file(path)

    if month in pre_months:
        pre_users |= users_m
    elif month in post_months:
        post_users |= users_m

In [ ]:
evacuated_users     = pre_users - post_users
non_evacuated_users = pre_users & post_users

In [ ]:
print("Pre users:", len(pre_users))
print("Post users:", len(post_users))
print("Evacuated:", len(evacuated_users))
print("Non-evacuated:", len(non_evacuated_users))

In [ ]:
user_evac_df = pd.DataFrame({
    "user": list(pre_users)
})

user_evac_df["evacuation"] = user_evac_df["user"].apply(
    lambda u: 1 if u in evacuated_users else 0
)
user_evac_df

In [ ]:
week_dir = os.path.join(graph_poi_dir, "user_evacuations" )
os.makedirs(week_dir, exist_ok=True)
output_path = os.path.join(week_dir, f"user_evacuations_{revision}.csv")
user_evac_df.to_csv(output_path)

In [ ]:
dfs = []
for month, path in month_files.items():
    if not os.path.exists(path):
        print(f"⚠️ Skipping missing {month}: {path}")
        continue

    print(f"📖 Loading {month} ...")
    df_m = load_month_jsonl_to_long(path, month)
    print("   rows:", len(df_m))
    dfs.append(df_m)

dyad_poi_all = pd.concat(dfs, ignore_index=True)
print("✅ Combined long df:", dyad_poi_all.shape)

In [ ]:
# -----------------------------
# A) LONG -> WIDE (monthly columns)
# -----------------------------
dyad_poi_wide = (
    dyad_poi_all
    .pivot_table(
        index=["user_i", "user_j", "poi"],
        columns="month",
        values="weight",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# month columns present
month_cols = [c for c in dyad_poi_wide.columns if c not in ["user_i", "user_j", "poi"]]
print("✅ dyad_poi_wide:", dyad_poi_wide.shape)
print("Month cols:", month_cols)

# -----------------------------
# B) Pre/Post sums (keep monthly columns)
# -----------------------------
pre_months  = ["Oct2021", "Nov2021"]
post_months = ["Jan2022", "Feb2022"]

# ensure missing month cols exist (if a month had zero rows, it won't appear in pivot)
for m in pre_months + post_months:
    if m not in dyad_poi_wide.columns:
        dyad_poi_wide[m] = 0

dyad_poi_wide["pre_sum"]  = dyad_poi_wide[pre_months].sum(axis=1)
dyad_poi_wide["post_sum"] = dyad_poi_wide[post_months].sum(axis=1)
dyad_poi_wide["delta_post_minus_pre"] = np.where(
    dyad_poi_wide["pre_sum"] > 0,
    (dyad_poi_wide["post_sum"] - dyad_poi_wide["pre_sum"]) / dyad_poi_wide["pre_sum"],
    np.nan
)

dyad_poi_wide["is_new_post"] = (
    (dyad_poi_wide["pre_sum"] == 0) &
    (dyad_poi_wide["post_sum"] > 0)
)

dyad_poi_wide["lost_in_post"] = (
    (dyad_poi_wide["post_sum"] == 0) &
    (dyad_poi_wide["pre_sum"] > 0)
)

print("✅ with pre/post sums:", dyad_poi_wide.shape)
dyad_poi_wide.sort_values("post_sum", ascending=False).head(5)

In [ ]:
week_dir = os.path.join(graph_poi_dir, "user_connections" )
os.makedirs(week_dir, exist_ok=True)
output_path = os.path.join(week_dir, f"connection_dynamics_poi_level_{revision}.csv")
dyad_poi_wide.to_csv(output_path)

# Load and merge pois meta data

In [ ]:
pre_edge_df_filename = f"CO_pois_TP.csv"
pre_edge_df_path = os.path.join(pois_dir, pre_edge_df_filename)
pois = pd.read_csv(pre_edge_df_path)
#pre_df["week_phase_label"] = "PDMW1" 
pois.columns

In [ ]:
output_path = os.path.join(week_dir, f"connection_dynamics_poi_level_{revision}.csv")
dyad_poi_wide = pd.read_csv(output_path)
dyad_poi_wide.head()

In [ ]:
# Merge dyad_poi_wide (poi) + pois (PLACEKEY) as LEFT JOIN
# (preserve all dyad_poi_wide rows)

# 1) Make sure join keys are same dtype + cleaned
dyad_poi_wide["poi"] = dyad_poi_wide["poi"].astype(str).str.strip()
pois["PLACEKEY"]     = pois["PLACEKEY"].astype(str).str.strip()

# (optional) quick sanity checks
print("dyad_poi_wide unique POIs:", dyad_poi_wide["poi"].nunique())
print("pois unique PLACEKEY:", pois["PLACEKEY"].nunique())
print("overlap keys:", dyad_poi_wide["poi"].isin(pois["PLACEKEY"]).mean())


In [ ]:
poi_keep_cols = [
    "PLACEKEY",
    "LATITUDE", "LONGITUDE",
    "STREET_ADDRESS", "CITY", "REGION", "POSTAL_CODE",
    "OVERALL_CATEGORY",
    "geometry", "point_geometry", "polygon_geometry",
    "index_right", "Unnamed: 0",
    "geography_id", "geography_name",
    "geometry_wkt", "centroid_lat", "centroid_lng",
    "fips_code",
    "TOP_CATEGORY", "SUB_CATEGORY",
]

In [ ]:
pois_sub = pois[poi_keep_cols].copy()


In [ ]:
#If PLACEKEY is not unique in pois, dedupe before merge (keeps first occurrence)
#    (This prevents row multiplication in the left join.)
dup_placekeys = pois_sub.duplicated("PLACEKEY").sum()
print("Duplicate PLACEKEY rows in pois_sub:", dup_placekeys)

if dup_placekeys > 0:
    pois_sub = pois_sub.drop_duplicates(subset="PLACEKEY", keep="first")

In [ ]:
# Left merge (preserve all dyad_poi_wide rows)
dyad_poi_wide_merged = dyad_poi_wide.merge(
    pois_sub,
    how="left",
    left_on="poi",
    right_on="PLACEKEY",
    validate="m:1"   # many dyad rows to one POI row; will error if pois_sub still has duplicates
)


In [ ]:
dyad_poi_wide_merged.shape

In [ ]:
print("✅ merged shape:", dyad_poi_wide_merged.shape)

#  Match diagnostics
match_rate = dyad_poi_wide_merged["PLACEKEY"].notna().mean()
print(f"✅ POI metadata match rate: {match_rate:.3f}")

unmatched = dyad_poi_wide_merged.loc[dyad_poi_wide_merged["PLACEKEY"].isna(), "poi"].nunique()
print("Unmatched POIs (nunique):", unmatched)

dyad_poi_wide_merged = dyad_poi_wide_merged.dropna(subset=["PLACEKEY"])


In [ ]:
dyad_poi_wide_merged.shape

In [ ]:
out_path = os.path.join(week_dir, f"connection_dynamics_poi_level_WITH_POI_META_{revision}.csv")
dyad_poi_wide_merged.to_csv(out_path, index=False)
print("Saved:", out_path)

# Individual level connection dynamics - statistics and removal of evacuaees

In [ ]:
revision = "R11"

In [ ]:
out_path = os.path.join(week_dir, f"connection_dynamics_poi_level_WITH_POI_META_{revision}.csv")
dyad_poi_wide = pd.read_csv(out_path)
dyad_poi_wide.head()

In [ ]:
dyad_poi_wide.columns

In [ ]:
# week_dir = os.path.join(graph_poi_dir, "user_evacuations" )
# os.makedirs(week_dir, exist_ok=True)
# output_path = os.path.join(week_dir, f"user_evacuations_{revision}.csv")
# user_evac_df = pd.read_csv(output_path)
# user_evac_df

In [ ]:
# df_nonevac_users = (
#     user_evac_df
#     .loc[user_evac_df["evacuation"] == 0, ["user"]]
#     .copy()
# )

In [ ]:
# df_nonevac_users.head()

In [ ]:
# Ensure consistent dtype (very important)
dyad_poi_wide["user_i"] = dyad_poi_wide["user_i"].astype(str)
dyad_poi_wide["user_j"] = dyad_poi_wide["user_j"].astype(str)
# df_nonevac_users["user"] = df_nonevac_users["user"].astype(str)

# # Build set of non-evacuated users
# nonevac_set = set(df_nonevac_users["user"])

# # Filter dyads where BOTH users are non-evacuated
# dyad_poi_wide = dyad_poi_wide[
#     dyad_poi_wide["user_i"].isin(nonevac_set) &
#     dyad_poi_wide["user_j"].isin(nonevac_set)
# ].copy()

print("✅ Rows before:", len(dyad_poi_wide))
#print("✅ Rows after filtering non-evac dyads:", len(dyad_poi_wide_merged))

In [ ]:
dyad_poi_wide["fips_code"] = dyad_poi_wide["fips_code"].apply(
    lambda x: str(int(x)) if pd.notnull(x) else pd.NA
)
dyad_poi_wide['fips_code']

## Include Home for each User

In [ ]:
start_date = 20211001
end_date = 20211131
freq_home_path = os.path.join(home_dir, f"freq_home_{start_date}_{end_date}")
home_df = pd.read_csv(freq_home_path)
home_df.columns

In [ ]:
home_df['geometry'] = home_df['geometry_wkt'].apply(wkt.loads)
home_df = gpd.GeoDataFrame(home_df, geometry='geometry', crs='EPSG:4326')
home_df["cbg_centroid"] = home_df.geometry.centroid
home_df = home_df[["cuebiq_id", "fips_code", "cbg_centroid", 'geometry']].copy()
home_df.columns

In [ ]:
home_df = home_df.rename(columns={"cuebiq_id": "user", "fips_code": "pre_disaster_home"})
home_df["user"] = home_df["user"].astype(str)
dyad_poi_wide["user_i"] = dyad_poi_wide["user_i"].astype(str)
dyad_poi_wide["user_j"] = dyad_poi_wide["user_j"].astype(str)

In [ ]:
dyad_poi_wide_merged_i = dyad_poi_wide.merge(home_df, left_on="user_i", right_on="user",how="left")
dyad_poi_wide_merged_i = dyad_poi_wide_merged_i.rename(columns={"pre_disaster_home": "pre_disaster_home_i",  
                                                                "cbg_centroid": "cbg_centroid_i", 'geometry': 'geometry_i'})
dyad_poi_wide_merged_i.columns

In [ ]:
dyad_poi_wide_merged_ij = dyad_poi_wide_merged_i.merge(home_df, left_on="user_j", right_on="user",how="left")
dyad_poi_wide_merged_ij = dyad_poi_wide_merged_ij.rename(columns={"pre_disaster_home": "pre_disaster_home_j",  
                                                                "cbg_centroid": "cbg_centroid_j", 'geometry': 'geometry_j'})

In [ ]:
dyad_poi_wide_merged_ij.columns

In [ ]:
# Boolean mask: missing either side
mask_missing_home = (
    dyad_poi_wide_merged_ij["pre_disaster_home_i"].isna() |
    dyad_poi_wide_merged_ij["pre_disaster_home_j"].isna()
)

n_missing_dyads = mask_missing_home.sum()
n_total_dyads = len(dyad_poi_wide_merged_ij)

print("Dyads with missing home (either side):", n_missing_dyads)
print("Total dyads:", n_total_dyads)
print("Share missing:", n_missing_dyads / n_total_dyads)

In [ ]:
missing_i = dyad_poi_wide_merged_ij["pre_disaster_home_i"].isna().sum()
missing_j = dyad_poi_wide_merged_ij["pre_disaster_home_j"].isna().sum()

print("Dyads missing home_i:", missing_i)
print("Dyads missing home_j:", missing_j)

In [ ]:
users_missing_i = set(
    dyad_poi_wide_merged_ij.loc[
        dyad_poi_wide_merged_ij["pre_disaster_home_i"].isna(),
        "user_i"
    ]
)

users_missing_j = set(
    dyad_poi_wide_merged_ij.loc[
        dyad_poi_wide_merged_ij["pre_disaster_home_j"].isna(),
        "user_j"
    ]
)

users_missing = users_missing_i.union(users_missing_j)

print("Unique users missing home:", len(users_missing))

In [ ]:
all_users_in_dyads = set(
    pd.concat([
        dyad_poi_wide_merged_ij["user_i"],
        dyad_poi_wide_merged_ij["user_j"]
    ])
)

print("Total unique users in dyads:", len(all_users_in_dyads))
print("Share missing users:", len(users_missing) / len(all_users_in_dyads))

In [ ]:
# Drop dyads where either user has missing pre-disaster home
dyad_clean = dyad_poi_wide_merged_ij.loc[
    ~(
        dyad_poi_wide_merged_ij["pre_disaster_home_i"].isna() |
        dyad_poi_wide_merged_ij["pre_disaster_home_j"].isna()
    )
].copy()

print("Remaining dyads:", len(dyad_clean))

In [ ]:
print("Unique users after drop:",
      len(set(pd.concat([dyad_clean["user_i"],
                         dyad_clean["user_j"]]))))

## Compute and include distance of poi from fire

In [ ]:
cbg_stats = os.path.join(base, "for_ABM/cbg_stats_census.csv")
cbg_stats_census = pd.read_csv(cbg_stats)
cbg_dist_fire = cbg_stats_census[["fips_code", "dist_epicentre"]].copy()
cbg_dist_fire = cbg_dist_fire.rename(columns={"fips_code": "poi_cbg"})
cbg_dist_fire.head()

In [ ]:
dyad_poi_wide_merged_ij.columns

In [ ]:
cbg_dist_fire["poi_cbg"] = cbg_dist_fire["poi_cbg"].astype(str)
# dyad_poi_wide_merged_ij["fips_code"] = dyad_poi_wide_merged["fips_code"].apply(
#     lambda x: str(int(x)) if pd.notnull(x) else pd.NA
# )
# dyad_poi_wide_merged_ij["fips_code"]
dyad_poi_wide_merged_ij = dyad_poi_wide_merged_ij.merge(cbg_dist_fire, left_on ="fips_code" , right_on = "poi_cbg", how = "left")
dyad_poi_wide_merged_ij = dyad_poi_wide_merged_ij.rename(columns={"dist_epicentre": "fire_dist_poi"})
dyad_poi_wide_merged_ij.columns

In [ ]:
dyad_poi_wide_merged_ij["fire_dist_poi"] = dyad_poi_wide_merged_ij["fire_dist_poi"].astype(int)
bins = pd.qcut(dyad_poi_wide_merged_ij["fire_dist_poi"], q=6, retbins=True, duplicates="drop")[1]

dyad_poi_wide_merged_ij["fire_distance_bin"] = pd.cut(
    dyad_poi_wide_merged_ij["fire_dist_poi"],
    bins=bins,
    include_lowest=True
)
dyad_poi_wide_merged_ij.head()

In [ ]:
cbg_stats = os.path.join(base, "for_ABM/cbg_stats_census.csv")
cbg_stats_census = pd.read_csv(cbg_stats)
cbg_dist_fire = cbg_stats_census[["fips_code", "dist_epicentre"]].copy()
cbg_dist_fire = cbg_dist_fire.rename(columns={"fips_code": "cbg"})
cbg_dist_fire.head()

In [ ]:
cbg_dist_fire["cbg"] = cbg_dist_fire["cbg"].astype(str)
dyad_poi_wide_merged_ij["pre_disaster_home_i"] = dyad_poi_wide_merged_ij["pre_disaster_home_i"].apply(
    lambda x: str(int(x)) if pd.notnull(x) else pd.NA
)

dyad_poi_wide_merged_ij = dyad_poi_wide_merged_ij.merge(cbg_dist_fire, left_on ="pre_disaster_home_i" , right_on = "cbg", how = "left")
dyad_poi_wide_merged_ij = dyad_poi_wide_merged_ij.rename(columns={"dist_epicentre": "fire_dist_home_i"})


In [ ]:
dyad_poi_wide_merged_ij["pre_disaster_home_j"] = dyad_poi_wide_merged_ij["pre_disaster_home_j"].apply(
    lambda x: str(int(x)) if pd.notnull(x) else pd.NA
)

dyad_poi_wide_merged_ij = dyad_poi_wide_merged_ij.merge(cbg_dist_fire, left_on ="pre_disaster_home_j" , right_on = "cbg", how = "left")
dyad_poi_wide_merged_ij = dyad_poi_wide_merged_ij.rename(columns={"dist_epicentre": "fire_dist_home_j"})


In [ ]:
dyad_poi_wide_merged_ij.columns

In [ ]:
dyad_poi_wide_merged_ij["edge_poi_dist_mean"] = (dyad_poi_wide_merged_ij["fire_dist_home_i"]+ dyad_poi_wide_merged_ij["fire_dist_home_j"])/2

In [ ]:
dyad_poi_wide_merged_ij.columns

In [ ]:
interactions = os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}.csv")
dyad_poi_wide_merged_ij.to_csv(interactions, index = False)


# CBG Level Stats

In [ ]:
revision = "R10_2"

In [ ]:
interactions = os.path.join(base, "for_ABM", f"individual_interac_nonevac_{revision}.csv")
dyad_poi_wide_merged_ij = pd.read_csv(interactions)

In [ ]:
cbg_agg = (
    dyad_poi_wide_merged_ij
    .groupby(
        ["fips_code", "OVERALL_CATEGORY"],
        as_index=False
    )
    .agg({
        # summed visit metrics
        "Oct2021": "sum",
        "Nov2021": "sum",
        "Jan2022": "sum",
        "Feb2022": "sum",
        "pre_sum": "sum",
        "post_sum": "sum",

        # averaged continuous covariates
        "edge_poi_dist_mean": "mean",

        # folded categorical / ID attributes
        "TOP_CATEGORY": list,
        "SUB_CATEGORY": list,
        "poi": list,

        # fire exposure (constant within CBG)
        "fire_distance_bin": "first",
        "fire_dist_poi": "first"
    })
)

print("✅ CBG × category aggregated shape:", cbg_agg.shape)
cbg_agg.columns

In [ ]:
# Poi level delta
cbg_agg["cbg_delta_interaction"] = (cbg_agg["post_sum"] - cbg_agg["pre_sum"]) / cbg_agg["pre_sum"]
cbg_agg["cbg_difference"] = (cbg_agg["post_sum"] - cbg_agg["pre_sum"])
cbg_agg["log_delta"] = np.log1p(cbg_agg["post_sum"]) - np.log1p(cbg_agg["pre_sum"])
cbg_agg["log_pre"] = np.log1p(cbg_agg["pre_sum"])
cbg_agg["log_post"] = np.log1p(cbg_agg["post_sum"])

In [ ]:
vals = cbg_agg["cbg_delta_interaction"].dropna()

plt.figure(figsize=(7,4))

sns.histplot( vals, bins=100, kde=True,color="purple",alpha=0.7)

plt.axvline(0, color="black", linestyle="--", linewidth=1)
# plt.title("Distribution of prob_shift_joint", fontsize=14)
# plt.xlabel("prob_shift_joint ( (post - pre) / pre )", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
interactions = os.path.join(base, "for_ABM", f"cbg_agg_interac_nonevac_{revision}.csv")
cbg_agg.to_csv(interactions, index=False)

# Poi level stats

In [ ]:
revision = "R11"

In [ ]:
interactions = os.path.join(base, "for_ABM", f"individual_interac_nonevac_{revision}.csv")
dyad_poi_wide_merged_ij = pd.read_csv(interactions)

In [ ]:
poi_agg = (
    dyad_poi_wide_merged_ij
    .groupby("poi", as_index=False)
    .agg({
        # summed visit metrics
        "Oct2021": "sum",
        "Nov2021": "sum",
        "Jan2022": "sum",
        "Feb2022": "sum",
        "pre_sum": "sum",
        "post_sum": "sum",
        "edge_poi_dist_mean": "mean",

        # folded categorical attributes
        'TOP_CATEGORY': "first",
        "fire_distance_bin": "first",
        "fire_dist_poi": "first",
        "OVERALL_CATEGORY": "first",
        "SUB_CATEGORY": "first",
        "fips_code": "first",
        "geometry_wkt": "first",
        "centroid_lat": "first", 
        "centroid_lng": "first"
    })
)

print("✅ POI aggregated shape:", poi_agg.shape)

poi_agg.columns

In [ ]:
# Poi level delta
eps = 1.0
pre = poi_agg["pre_sum"].astype(float)
post = poi_agg["post_sum"].astype(float)
poi_agg["poi_delta_interaction"] = (poi_agg["post_sum"] - poi_agg["pre_sum"]) / poi_agg["pre_sum"]
poi_agg["poi_difference"] = (poi_agg["post_sum"] - poi_agg["pre_sum"])
poi_agg["log_pre"]  = np.log10(pre + eps)
poi_agg["log_post"] = np.log10(post + eps)
poi_agg["log_delta"] = poi_agg["log_post"] - poi_agg["log_pre"]
poi_agg["ln_pre"] = np.log(pre + 1.0)
poi_agg["ln_post"] = np.log(post + 1.0)
poi_agg["fire_dist_km"] = poi_agg["fire_dist_poi"] / 1000
poi_agg["edge_poi_dist_mean_km"] = poi_agg["edge_poi_dist_mean"] / 1000

In [ ]:
poi_agg.head()

## Include Census

In [ ]:
census_path = os.path.join(geo_dir, "colorado_cbg_census_only.csv")
census = pd.read_csv(census_path)
census["black_percent"] = census["black_alone"] / census["total_population"]
census["white_percent"] = census["white_alone"] / census["total_population"]
census.loc[census["median_income"] < 0, "median_income"] = 1
census.loc[census["median_rent"] < 0, "median_rent"] = 1
census["median_income_log"] = np.log1p(census["median_income"])
census.head()

In [ ]:
census["cbg_fips"] = census["cbg_fips"].astype(str)
poi_agg["fips_code"] = poi_agg["fips_code"].astype(str)

In [ ]:
poi_agg_census = poi_agg.merge(census,
    left_on='fips_code',  right_on='cbg_fips',
    how='left')
poi_agg_census.head()

In [ ]:
interactions = os.path.join(base, "for_ABM", f"poi_agg_interac_nonevac_{revision}.csv")
poi_agg_census.to_csv(interactions, index = False)
print(interactions)

# Regressions - POI

In [ ]:
revision = "R11"
interactions = os.path.join(base, "for_ABM", f"poi_agg_interac_nonevac_{revision}.csv")
poi_agg_census = pd.read_csv(interactions)
poi_agg_census.head()

In [ ]:
poi_agg_census.columns

In [ ]:
vals = poi_agg_census["ln_pre"].dropna()

plt.figure(figsize=(7,4))

sns.histplot( vals, bins=100, kde=True,color="purple",alpha=0.7)
#plt.yscale("log")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.title(f"Distribution of prob_shift_joint - {revision}", fontsize=14)
# plt.xlabel("prob_shift_joint ( (post - pre) / pre )", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()
plt.show()

### Just Sanity checks - ignore

In [ ]:
poi_agg_census_2000 = poi_agg_census[
    (poi_agg_census["poi"].notna()) &
    (poi_agg_census["pre_sum"] > 5000)]
poi_agg_census_2000.shape

In [ ]:
poi_agg_census_2000["topcat_freq_at_coord"] = (
    poi_agg_census_2000
    .groupby(["centroid_lat", "centroid_lng", "SUB_CATEGORY"])["poi"]
    .transform("count")
)

In [ ]:
poi_agg_census_2000["activity_score"] = poi_agg_census_2000["pre_sum"] + poi_agg_census_2000["post_sum"]

In [ ]:
poi_agg_census_2000 = poi_agg_census_2000.sort_values(
    by=[
        "centroid_lat",
        "centroid_lng",
        "topcat_freq_at_coord",  # ← THIS is now correct
        "activity_score",
    ],
    ascending=[
        True,   # lat
        True,   # lng
        True,   # rarer category first
        False,  # higher activity preferred
    ]
)


In [ ]:
poi_agg_dedup = (
    poi_agg_census_2000
    .drop_duplicates(
        subset=["centroid_lat", "centroid_lng"],
        keep="first"
    )
    .copy()
)
poi_agg_dedup.shape

In [ ]:
poi_agg_census["poi"] = poi_agg_census["poi"].astype(str)
poi_agg_census_2000["poi"] = poi_agg_census_2000["poi"].astype(str)
poi_agg_dedup["poi"] = poi_agg_dedup["poi"].astype(str)

In [ ]:
high_poi_ids = set(poi_agg_census_2000["poi"])

poi_agg_rest = poi_agg_census.loc[
    ~poi_agg_census["poi"].isin(high_poi_ids)
].copy()

print("Remaining (non-high-activity) POIs:", poi_agg_rest.shape)

In [ ]:
poi_agg_final = pd.concat(
    [poi_agg_rest, poi_agg_dedup],
    ignore_index=True
)

In [ ]:
poi_agg_final.head()

## Regression Loop

In [ ]:
def extract_model_results(model, model_name):
    """
    Returns a tidy dataframe with:
    Variable | Coefficient | P_value | CI_low | CI_high | R2 | Model
    """
    coef = model.params
    pval = model.pvalues

    # confidence intervals
    ci = model.conf_int()
    ci.columns = ["CI_low", "CI_high"]

    # R2 for OLS, pseudo-R2 for Logit
    if hasattr(model, "rsquared"):
        r2 = float(model.rsquared)
    else:
        r2 = float(model.prsquared)

    # IMPORTANT: align by variable index so CI doesn't become NaN
    df = pd.DataFrame(
        {
            "Variable": coef.index,
            "Coefficient": coef.values,
            "P_value": pval.values,
            "CI_low": ci.loc[coef.index, "CI_low"].values,
            "CI_high": ci.loc[coef.index, "CI_high"].values,
        }
    ).assign(
        Coefficient=lambda d: d["Coefficient"].round(3),
        P_value=lambda d: d["P_value"].round(3),
        CI_low=lambda d: d["CI_low"].round(3),
        CI_high=lambda d: d["CI_high"].round(3),
        R2=round(r2, 3),
        Model=model_name,
    )

    return df

In [ ]:
sdm_cols = [
    "total_population", "median_age",
    "white_percent", 
    #"black_alone",
    "median_income_log", "median_rent", 
]

sdm_terms = " + ".join(sdm_cols)

In [ ]:
poi_agg_final["TOP_CATEGORY"].unique()

In [ ]:
poi_agg_final.columns

In [ ]:
from sklearn.preprocessing import StandardScaler

std_cols = ["fire_dist_poi","edge_poi_dist_mean"] + sdm_cols
poi_std = poi_agg_final.copy()

scaler = StandardScaler()
poi_std[std_cols] = scaler.fit_transform(poi_std[std_cols])


In [ ]:
# List of (model_name, formula)
ref_cat = "Amusement Parks and Arcades"

model_specs = [
    ("pre_base",  "log_pre ~ fire_dist_poi + edge_poi_dist_mean"),
    ("diff_base", "log_delta ~ fire_dist_poi + edge_poi_dist_mean"),

    ("pre_cat",
     f'log_pre ~ fire_dist_poi + edge_poi_dist_mean + C(TOP_CATEGORY, Treatment(reference="{ref_cat}"))'),

    ("diff_cat",
     f'log_delta ~ fire_dist_poi + edge_poi_dist_mean + C(TOP_CATEGORY, Treatment(reference="{ref_cat}"))'),

    ("pre_cat_sdm",
     f'log_pre ~ fire_dist_poi + edge_poi_dist_mean + C(TOP_CATEGORY, Treatment(reference="{ref_cat}")) + {sdm_terms}'),

    ("diff_cat_sdm",
     f'log_delta ~ fire_dist_poi + edge_poi_dist_mean + C(TOP_CATEGORY, Treatment(reference="{ref_cat}")) + {sdm_terms}'),
    
    ("pre_dist_sdm",
     f'log_pre ~ fire_dist_poi + edge_poi_dist_mean  + {sdm_terms}'),

    ("diff_dist_sdm",
     f'log_delta ~ fire_dist_poi + edge_poi_dist_mean + {sdm_terms}'),
]

In [ ]:
all_results = []
fitted_models = {}

for model_name, formula in model_specs:
    print(f"Running model: {model_name}")

    model = sm.OLS.from_formula(
        formula=formula,
        data=poi_std
    ).fit()

    fitted_models[model_name] = model  # optional: keep models

    res_df = extract_model_results(model, model_name)
    all_results.append(res_df)

In [ ]:
results_df = pd.concat(all_results, ignore_index=True)
results_edges_all = os.path.join(base, "for_ABM", "reg_edges_all.csv")
results_df.to_csv(results_edges_all, index=False)


## Plot regression results

In [ ]:
results_edges_all = os.path.join(base, "for_ABM", "reg_edges_all.csv")
results_df_all = pd.read_csv(results_edges_all)

In [ ]:
results_df_all.columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_coef_with_ci_and_r2_legend(
    results_df,
    *,
    var_order=None,
    var_pretty=None,
    model_pretty=None,
    # subset control
    models_to_plot=None,          # e.g. ["pre_cat_sdm", "diff_cat_sdm"]
    match_on="raw",               # "raw" (Model) or "pretty" (Model_pretty) or "either"
    # significance / styling
    alpha_sig=0.05,               # used if "significant" col missing
    grey_non_sig=True,
    nonsig_color="lightgrey",
    nonsig_alpha=0.35,
    sig_alpha=0.95,
    # plot styling
    tick_fontsize=12,
    marker_size=70,
    ci_lw=1.6,
    figsize=(12, 7),
    drop_intercept=True,
    legend_loc="upper left",
    legend_bbox_to_anchor=(1.02, 1.0),
    legend_fontsize=14,
    pre_color=None,
    delta_color=None,
):
    """
    Single-panel coefficient plot:
      - each model has different marker
      - ONLY TWO colors: one for Pre models and one for Δ models
      - confidence intervals are grey + dotted
      - legend includes R² in brackets
      - OPTIONAL: grey out non-significant points (and CIs) using results_df["significant"]
        (or computed from P_value < alpha_sig if missing)
    Expects columns: Variable, Coefficient, CI_low, CI_high, R2, Model
    Optional columns: significant (bool), P_value
    """
    df = results_df.copy()

    # pretty maps
    var_pretty = {} if var_pretty is None else var_pretty
    model_pretty = {} if model_pretty is None else model_pretty

    df["Variable_pretty"] = df["Variable"].map(var_pretty).fillna(df["Variable"])
    df["Model_pretty"] = df["Model"].map(model_pretty).fillna(df["Model"])

    # filter models
    if models_to_plot is not None:
        wanted = set(map(str, models_to_plot))
        if match_on == "raw":
            df = df[df["Model"].astype(str).isin(wanted)].copy()
        elif match_on == "pretty":
            df = df[df["Model_pretty"].astype(str).isin(wanted)].copy()
        elif match_on == "either":
            df = df[
                df["Model"].astype(str).isin(wanted) |
                df["Model_pretty"].astype(str).isin(wanted)
            ].copy()
        else:
            raise ValueError("match_on must be one of {'raw','pretty','either'}")

    if df.shape[0] == 0:
        raise ValueError("After filtering, no rows remain. Check models_to_plot/match_on values.")

    # significance flag (use your precomputed column if present)
    if "significant" not in df.columns:
        if "P_value" not in df.columns:
            raise ValueError("Need either a boolean 'significant' column or a numeric 'P_value' column.")
        df["significant"] = df["P_value"].astype(float) < float(alpha_sig)

    if drop_intercept:
        df = df[~df["Variable"].isin(["Intercept", "const"])].copy()

    # variable order (not by coefficient)
    if var_order is None:
        ordered_vars = df["Variable_pretty"].drop_duplicates().tolist()
    else:
        base = [var_pretty.get(v, v) for v in var_order]
        rest = [v for v in df["Variable_pretty"].drop_duplicates().tolist() if v not in base]
        ordered_vars = base + rest
    df["Variable_pretty"] = pd.Categorical(df["Variable_pretty"], categories=ordered_vars, ordered=True)

    # model list
    models = df["Model_pretty"].drop_duplicates().tolist()

    # two-color scheme (Pre vs Δ) + unique markers per model
    if pre_color is None or delta_color is None:
        two = sns.color_palette(n_colors=2)
        if pre_color is None:
            pre_color = two[0]
        if delta_color is None:
            delta_color = two[1]

    def is_delta(model_label: str) -> bool:
        s = str(model_label)
        return ("Δ" in s) or s.lower().startswith(("diff", "delta"))

    base_color_map = {m: (delta_color if is_delta(m) else pre_color) for m in models}

    markers = ["o", "s", "D", "^", "v", "P", "X", "*", "<", ">", "h", "H"]
    marker_map = {m: markers[i % len(markers)] for i, m in enumerate(models)}

    # compute R2 per model
    r2_by_model = df.groupby("Model_pretty")["R2"].max()
    legend_label = {m: f"{m} (R²={r2_by_model.loc[m]:.3f})" for m in models}

    # plot
    fig, ax = plt.subplots(figsize=figsize)

    y_levels = df["Variable_pretty"].cat.categories.tolist()
    y_pos = {v: i for i, v in enumerate(y_levels)}
    df["_y"] = df["Variable_pretty"].map(y_pos).astype(float)

    offsets = np.linspace(-0.20, 0.20, num=max(2, len(models)))
    model_offset = {m: offsets[i] for i, m in enumerate(models)}
    df["_y_dodge"] = df["_y"] + df["Model_pretty"].map(model_offset)

    handles, labels = [], []

    for m in models:
        sub = df[df["Model_pretty"] == m].copy()

        # split sig/nonsig for styling
        sub_sig = sub[sub["significant"]].copy()
        sub_nsig = sub[~sub["significant"]].copy()

        # --- CIs ---
        # significant CIs (always grey dotted)
        if len(sub_sig):
            ax.hlines(
                y=sub_sig["_y_dodge"],
                xmin=sub_sig["CI_low"],
                xmax=sub_sig["CI_high"],
                linewidth=ci_lw,
                color="grey",
                linestyle=":",
                alpha=0.85,
                zorder=1,
            )

        # non-significant CIs (greyed out more)
        if len(sub_nsig):
            ax.hlines(
                y=sub_nsig["_y_dodge"],
                xmin=sub_nsig["CI_low"],
                xmax=sub_nsig["CI_high"],
                linewidth=ci_lw,
                color=nonsig_color if grey_non_sig else "grey",
                linestyle=":",
                alpha=nonsig_alpha if grey_non_sig else 0.85,
                zorder=1,
            )

        # --- points ---
        # non-significant points: grey
        if len(sub_nsig):
            ax.scatter(
                sub_nsig["Coefficient"],
                sub_nsig["_y_dodge"],
                s=marker_size,
                marker=marker_map[m],
                color=nonsig_color if grey_non_sig else base_color_map[m],
                alpha=nonsig_alpha if grey_non_sig else sig_alpha,
                edgecolor="none",
                zorder=2,
            )

        # significant points: colored by pre/delta
        if len(sub_sig):
            sc = ax.scatter(
                sub_sig["Coefficient"],
                sub_sig["_y_dodge"],
                s=marker_size,
                marker=marker_map[m],
                color=base_color_map[m],
                alpha=sig_alpha,
                edgecolor="none",
                zorder=3,
            )
        else:
            # keep legend handle even if no significant points exist in this model
            sc = ax.scatter([], [], s=marker_size, marker=marker_map[m], color=base_color_map[m])

        handles.append(sc)
        labels.append(legend_label[m])

    ax.axvline(0, color="black", linestyle="--", linewidth=1, zorder=0)

    ax.set_yticks(range(len(y_levels)))
    ax.set_yticklabels(y_levels)
    ax.tick_params(axis="x", labelsize=tick_fontsize)
    ax.tick_params(axis="y", labelsize=tick_fontsize)

    ax.set_title("")
    ax.set_xlabel("Coefficient", fontsize=tick_fontsize + 1)
    ax.set_ylabel("")
    sns.despine(ax=ax, left=True, bottom=False)

    ax.legend(
        handles,
        labels,
        title="Model",
        loc=legend_loc,
        bbox_to_anchor=legend_bbox_to_anchor,
        frameon=False,
        fontsize=legend_fontsize,
        title_fontsize=legend_fontsize,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
#results_df_all["Variable"].unique()

In [ ]:
import re

def pretty_variable_name(v: str) -> str:
    v = str(v)

    # Intercept
    if v in ["Intercept", "const"]:
        return "Intercept"

    # Your continuous vars
    base_map = {
        "fire_dist_poi": "Distance to fire (m)",
        "edge_poi_dist_mean": "Average distance from home (m)",
        "median_income_log": "Median income (log)",
        "median_rent": "Median rent",
        "white_percent": "% White",
        "total_population": "Total population",
        "median_age": "Median age",
    }
    if v in base_map:
        return base_map[v]

    # Category dummies from statsmodels: C(TOP_CATEGORY)[T.<level>]

    REF = "Amusement Parks and Arcades"

    pat = rf'^C\(TOP_CATEGORY,\s*Treatment\(reference=[\'"]{re.escape(REF)}[\'"]\)\)\[T\.(.*)\]$'
    m = re.match(pat, v)

    if m:
        level = m.group(1).strip()

        # Optional: shorten some common long phrases (edit as you like)
        level = level.replace("Restaurants and Other Eating Places", "Restaurants $ Cafes")
        level = level.replace("Sporting Goods, Hobby, and Musical Instrument Stores", "Sporting/Hobby/Music Stores")
        level = level.replace("Drinking Places (Alcoholic Beverages)", "Bars & Pubs")
        level = level.replace("Other Amusement and Recreation Industries", "Amusement & Recreation")
        level = level.replace("Performing Arts Companies", "Fitness and Performing Art studios")
        level = level.replace("Motion Picture and Video Industries", "Theatre and Cinema")
        level = level.replace("Museums, Historical Sites, and Similar Institutions", "Museums & Historical Sites")
        level = level.replace("Bakeries and Tortilla Manufacturing", "Bakeries")
        level = level.replace("Book Stores and News Dealers", "Bookstores & Newsstands")
        level = level.replace("Other Miscellaneous Store Retailers", "Other retail (Delis")
        level = level.replace("Personal Care Services", "Personal Care")
        level = level.replace("Specialty Food Stores", "Specialty Food Stores")

        return f"Category: {level}"

    # Fallback (keep as-is)
    return v


# Then build var_pretty dict automatically from what exists in your dataframe:
var_pretty = {v: pretty_variable_name(v) for v in results_df_all["Variable"].unique()}

In [ ]:
results_df_all["Model"].unique()


In [ ]:
results_df_all = pd.read_csv(results_edges_all)

# combine (if you want both sets on one plot)
df_plot = pd.concat([results_df_all], ignore_index=True)
results_df_all["significant"] = results_df_all["P_value"] < 1
model_pretty = {
    "pre_base": "Pre: Base",
    "diff_base": "Δ: Base",
    "pre_cat": "Pre: +Cat",
    "diff_cat": "Δ: +Cat",
    "pre_cat_sdm": "Pre: +Cat+SDM",
    "diff_cat_sdm": "Δ: +Cat+SDM",
}


plot_coef_with_ci_and_r2_legend(
    results_df_all,
    models_to_plot=["pre_cat", "diff_cat"],
    match_on="raw",
    var_pretty=var_pretty,
    model_pretty=model_pretty,
    tick_fontsize=20,
    figsize=(6, 9),
)

In [ ]:
results_df_all = pd.read_csv(results_edges_all)

# combine (if you want both sets on one plot)
df_plot = pd.concat([results_df_all], ignore_index=True)
results_df_all["significant"] = results_df_all["P_value"] < 0.3
model_pretty = {
    "pre_base": "Pre: Base",
    "diff_base": "Δ: Base",
    "pre_cat": "Pre: +Cat",
    "diff_cat": "Δ: +Cat",
    "pre_cat_sdm": "Pre: +Cat+SDM",
    "diff_cat_sdm": "Δ: +Cat+SDM",
}


plot_coef_with_ci_and_r2_legend(
    results_df_all,
    models_to_plot=["pre_dist_sdm", "diff_dist_sdm"],
    match_on="raw",
    var_pretty=var_pretty,
    model_pretty=model_pretty,
    tick_fontsize=20,
    figsize=(6, 6),
)

## Regressions

In [ ]:
# poi_agg_census_20 = poi_agg_final[
#     (poi_agg_final["poi"].notna()) &
#     (poi_agg_final["pre_sum"] > 0)]

In [ ]:
sdm_cols = [
    "total_population", "median_age",
    "white_percent", 
    #"black_alone",
    "median_income_log", "median_rent", 
]

sdm_terms = " + ".join(sdm_cols)

In [ ]:
from sklearn.preprocessing import StandardScaler

std_cols = ["fire_dist_poi","edge_poi_dist_mean"] + sdm_cols
poi_std = poi_agg_census.copy()

scaler = StandardScaler()
poi_std[std_cols] = scaler.fit_transform(poi_std[std_cols])


In [ ]:
poi_std.shape

In [ ]:
model_ols_delta_dist = sm.OLS.from_formula(
    "ln_pre ~ fire_dist_poi + edge_poi_dist_mean",
    data=poi_std
).fit()
print(model_ols_delta_dist.summary())

In [ ]:
poi_std["TOP_CATEGORY"].unique()

In [ ]:
model_ols_delta_cat = sm.OLS.from_formula(
    'log_pre ~ C(SUB_CATEGORY)',
    data=poi_std
).fit()
model_ols_delta_cat.summary()


In [ ]:
model_ols_delta_cat = sm.OLS.from_formula(
    "log_delta~ fire_dist_poi + edge_poi_dist_mean + SUB_CATEGORY",
    data=poi_std
).fit()
model_ols_delta_cat.summary()


In [ ]:
sdm_cols = [
    "total_population", #"median_age",
    "white_percent", 
    #"black_percent",
    "median_income_log", #"median_rent", 
]

sdm_terms = " + ".join(sdm_cols)
formula = (
    "poi_difference ~ fire_dist_poi + edge_poi_dist_mean +" + sdm_terms
)
model_ols_delta_dist_x_cat_sdm1 = sm.OLS.from_formula(
    formula,
    data=poi_std
).fit()
model_ols_delta_dist_x_cat_sdm1.summary()


## Regression, residual - observed vs expected

In [ ]:
sdm_cols = ["total_population","median_age","white_percent","median_income_log","median_rent"]
std_cols = ["fire_dist_poi", "edge_poi_dist_mean"] + sdm_cols

poi_agg_census = poi_agg_census.copy()

poi_std = poi_agg_census.copy()  # IMPORTANT: same index, same rows
scaler = StandardScaler()
poi_std[std_cols] = scaler.fit_transform(poi_std[std_cols])

In [ ]:
formula_pre   = 'log_pre   ~ C(SUB_CATEGORY) + ' + " + ".join(std_cols)
formula_delta = 'log_delta ~ C(SUB_CATEGORY) + ' + " + ".join(std_cols)

m_pre   = smf.ols(formula_pre,   data=poi_std).fit(cov_type="HC3")
m_delta = smf.ols(formula_delta, data=poi_std).fit(cov_type="HC3")

In [ ]:
# m_pre   = smf.ols('log_pre   ~ C(SUB_CATEGORY)', data=poi_std).fit()
# m_delta = smf.ols('log_delta ~ C(SUB_CATEGORY)', data=poi_std).fit()

In [ ]:
# poi_agg_census["yhat_log_pre_subcat"]   = m_pre.fittedvalues
# poi_agg_census["yhat_log_delta_subcat"] = m_delta.fittedvalues

In [ ]:
poi_agg_census["yhat_log_pre"]   = m_pre.predict(poi_std)
poi_agg_census["yhat_log_delta"] = m_delta.predict(poi_std)

# residuals on log scale:
poi_agg_census["resid_log_pre"]   = poi_std["log_pre"]   - poi_agg_census["yhat_log_pre"]
poi_agg_census["resid_log_delta"] = poi_std["log_delta"] - poi_agg_census["yhat_log_delta"]

In [ ]:
poi_agg_census["z_pre"]   = (poi_agg_census["resid_log_pre"]   - poi_agg_census["resid_log_pre"].mean()) / poi_agg_census["resid_log_pre"].std(ddof=1)
poi_agg_census["z_delta"] = (poi_agg_census["resid_log_delta"] - poi_agg_census["resid_log_delta"].mean()) / poi_agg_census["resid_log_delta"].std(ddof=1)

In [ ]:
poi_agg_census.head()

In [ ]:
vals = poi_agg_census["z_pre"].dropna()

plt.figure(figsize=(7,4))

sns.histplot( vals, bins=100, kde=True,color="purple",alpha=0.7)
#plt.yscale("log")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
#plt.title(f"Distribution of prob_shift_joint - {revision}", fontsize=14)
# plt.xlabel("prob_shift_joint ( (post - pre) / pre )", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
vals = poi_agg_census["z_delta"].dropna()

plt.figure(figsize=(7,4))

sns.histplot( vals, bins=100, kde=True,color="purple",alpha=0.7)
#plt.yscale("log")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
# plt.title(f"Distribution of prob_shift_joint - {revision}", fontsize=14)
# plt.xlabel("prob_shift_joint ( (post - pre) / pre )", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()
plt.show()

## Regression Plots

In [ ]:
def extract_model_results(model, model_name):
    """
    Returns a tidy dataframe with:
    Variable | Coefficient | P-value | R2 | Model
    """
    coef = model.params
    pval = model.pvalues

    # R2 for OLS, pseudo-R2 for Logit
    if hasattr(model, "rsquared"):
        r2 = model.rsquared
    else:
        r2 = model.prsquared

    df = (
        pd.DataFrame({
            "Variable": coef.index,
            "Coefficient": coef.values,
            "P_value": pval.values
        })
        .assign(
            Coefficient=lambda d: d["Coefficient"].round(3),
            P_value=lambda d: d["P_value"].round(3),
            R2=round(r2, 3),
            Model=model_name
        )
    )

    return df

In [ ]:
model_results = pd.concat([
    extract_model_results(model_ols_delta_dist, "OLS: log_delta ~ distance"),
    extract_model_results(model_ols_delta_cat, "OLS: log_delta ~ category"),
    extract_model_results(model_ols_delta_dist_cat, "OLS: log_delta ~ distance + category"),
    # extract_model_results(model_ols_delta_dist_x_cat, "OLS: log_delta ~ distance × category"),
    # extract_model_results(model_ols_delta_dist_x_cat_sdm1, "OLS: log_delta ~ distance × category + SDM"),
    extract_model_results(model_ols_delta_dist_cat_sdm2, "OLS: log_delta ~ distance + category + SDM"),

    extract_model_results(model_logit_drop_dist, "Logit: drop_post ~ distance"),
    extract_model_results(model_logit_drop_dist_cat, "Logit: drop_post ~ distance + category"),
    extract_model_results(model_logit_drop_cat_dist_sdm, "Logit: drop_post ~ distance + category + SDM"),

    extract_model_results(model_logit_new_dist, "Logit: new_post ~ distance"),
    extract_model_results(model_logit_new_dist_cat, "Logit: new_post ~ distance + category"),
    extract_model_results(model_logit_new_cat_dist_sdm, "Logit: new_post ~ distance + category + SDM"),
], ignore_index=True)

In [ ]:
var_rename = {
    "fire_dist_poi": "Distance to fire",
    "total_population": "Total population",
    "median_age": "Median age",
    "white_percent": "White (%)",
    "black_percent": "Black (%)",
    "median_income_log": "Median income (log)",
    "median_rent": "Median rent",

    "C(OVERALL_CATEGORY)[T.Dining - Mixed]": "Dining (Mixed)",
    "C(OVERALL_CATEGORY)[T.Dining - Restaurants, bars, pubs]": "Dining (Restaurants)",
    "C(OVERALL_CATEGORY)[T.Food Retail - Specialty]": "Food retail (Specialty)",
    "C(OVERALL_CATEGORY)[T.Recreation - Amusement]": "Recreation (Amusement)",
    "C(OVERALL_CATEGORY)[T.Recreation - Cinema]": "Recreation (Cinema)",
    "C(OVERALL_CATEGORY)[T.Retail - Bookstores]": "Retail (Bookstores)",
    "C(OVERALL_CATEGORY)[T.Retail - General & Specialty]": "Retail (General)",
}

plot_df["Variable_clean"] = plot_df["Variable"].map(var_rename).fillna(plot_df["Variable"])

In [ ]:
def sig_star(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.1:
        return "*"
    else:
        return ""

plot_df["stars"] = plot_df["P_value"].apply(sig_star)

In [ ]:
var_order = (
    plot_df["Variable_clean"]
    .value_counts()
    .index
    .tolist()
)

plot_df["Variable_clean"] = pd.Categorical(
    plot_df["Variable_clean"],
    categories=var_order,
    ordered=True
)

In [ ]:
plot_df["model_type"] = np.where(
    plot_df["Model"].str.contains("drop", case=False),
    "Lost POIs",
    np.where(
        plot_df["Model"].str.contains("new", case=False),
        "New POIs",
        "Other"
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

models = plot_df["Model"].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(models)))

y_positions = np.arange(len(var_order))

for model, color in zip(models, colors):
    df_m = plot_df[plot_df["Model"] == model]

    y = df_m["Variable_clean"].cat.codes
    x = df_m["Coefficient"]

    ax.scatter(
        x,
        y,
        s=60,
        color=color,
        alpha=0.85,
        label=model
    )

    # significance stars
    for xi, yi, star in zip(x, y, df_m["stars"]):
        if star:
            ax.text(
                xi,
                yi,
                f" {star}",
                va="center",
                ha="left",
                fontsize=10,
                color=color
            )

# zero line
ax.axvline(0, color="black", linestyle="--", linewidth=1)

ax.set_yticks(y_positions)
ax.set_yticklabels(var_order)
ax.set_xlabel("Coefficient estimate")
ax.set_title("Regression coefficients across models\n(insane coefficients removed)")

ax.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()

In [ ]:
def restrict_variable_axis(df):
    """
    Restrict Variable_clean categories to only those
    that appear in this dataframe.
    """
    present_vars = (
        df["Variable_clean"]
        .dropna()
        .unique()
        .tolist()
    )

    df = df.copy()
    df["Variable_clean"] = pd.Categorical(
        df["Variable_clean"],
        categories=present_vars,
        ordered=True
    )

    return df

In [ ]:
lost_df = restrict_variable_axis(
    plot_df[plot_df["Model"].str.contains("drop_post")]
)

new_df = restrict_variable_axis(
    plot_df[plot_df["Model"].str.contains("new_post")]
)

In [ ]:
def plot_coefficients(df, title):
    fig, ax = plt.subplots(figsize=(9, 4))

    models = df["Model"].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

    y_positions = np.arange(len(df["Variable_clean"].cat.categories))

    for model, color in zip(models, colors):
        d = df[df["Model"] == model]

        y = d["Variable_clean"].cat.codes
        x = d["Coefficient"]

        ax.scatter(
            x, y,
            s=60,
            color=color,
            alpha=0.85,
            label=model
        )

        # significance stars
        for xi, yi, star in zip(x, y, d["stars"]):
            if star:
                ax.text(
                    xi,
                    yi,
                    f" {star}",
                    va="center",
                    ha="left",
                    fontsize=10,
                    color=color
                )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(df["Variable_clean"].cat.categories)

    ax.set_xlabel("Coefficient estimate")
    ax.set_title(title)

    ax.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    plt.tight_layout()
    return fig, ax

In [ ]:
fig_lost, ax_lost = plot_coefficients(
    lost_df,
    title="Determinants of POI loss after disaster (drop_post)"
)

In [ ]:
fig_new, ax_new = plot_coefficients(
    new_df,
    title="Determinants of new POI emergence after disaster (new_post)"
)

In [ ]:
old_df = plot_df[
    plot_df["Model"].str.startswith("OLS:")
].copy()

old_df = restrict_variable_axis(old_df)

In [ ]:
fig_old, ax_old = plot_coefficients(
    old_df,
    title="Change in interaction intensity at existing POIs (log_delta)"
)

# Regression - CBG

In [ ]:
interactions = os.path.join(base, "for_ABM", "cbg_agg_interac_nonevac.csv")
cbg_agg = pd.read_csv(interactions)
cbg_agg.head()

In [ ]:
census_path = os.path.join(geo_dir, "colorado_cbg_census_only.csv")
census = pd.read_csv(census_path)
census["black_percent"] = census["black_alone"] / census["total_population"]
census["white_percent"] = census["white_alone"] / census["total_population"]
census.loc[census["median_income"] < 0, "median_income"] = 1
census.loc[census["median_rent"] < 0, "median_rent"] = 1
census["median_income_log"] = np.log1p(census["median_income"])
census.head()

In [ ]:
census["cbg_fips"] = census["cbg_fips"].astype(str)
cbg_agg["fips_code"] = cbg_agg["fips_code"].astype(str)

In [ ]:
cbg_agg_census = cbg_agg.merge(census,
    left_on='fips_code',  right_on='cbg_fips',
    how='left')


In [ ]:
cbg_agg_census.columns

In [ ]:
def extract_model_results(model, model_name):
    """
    Returns a tidy dataframe with:
    Variable | Coefficient | P-value | R2 | Model
    """
    coef = model.params
    pval = model.pvalues

    # R2 for OLS, pseudo-R2 for Logit
    if hasattr(model, "rsquared"):
        r2 = model.rsquared
    else:
        r2 = model.prsquared

    df = (
        pd.DataFrame({
            "Variable": coef.index,
            "Coefficient": coef.values,
            "P_value": pval.values
        })
        .assign(
            Coefficient=lambda d: d["Coefficient"].round(3),
            P_value=lambda d: d["P_value"].round(3),
            R2=round(r2, 3),
            Model=model_name
        )
    )

    return df

In [ ]:
sdm_cols = [
    "total_population", "median_age",
    "white_percent", 
    #"black_alone",
    "median_income_log", "median_rent", 
]

sdm_terms = " + ".join(sdm_cols)

In [ ]:
# List of (model_name, formula)
model_specs = [
    ("post_base","post_sum ~ fire_dist_poi + edge_poi_dist_mean"),
    ("pre_base","pre_sum ~ fire_dist_poi + edge_poi_dist_mean"),
    ("diff_base","cbg_difference ~ fire_dist_poi + edge_poi_dist_mean"),
    ("post_cat","post_sum ~ fire_dist_poi + edge_poi_dist_mean + C(OVERALL_CATEGORY)"),
    ("pre_cat","pre_sum ~ fire_dist_poi + edge_poi_dist_mean + C(OVERALL_CATEGORY)"),
    ("diff_cat", "cbg_difference ~ fire_dist_poi + edge_poi_dist_mean + C(OVERALL_CATEGORY)"),
    ("post_cat_sdm", "post_sum ~ fire_dist_poi + edge_poi_dist_mean + C(OVERALL_CATEGORY) + " + sdm_terms),
    ("pre_cat_sdm","pre_sum ~ fire_dist_poi + edge_poi_dist_mean + C(OVERALL_CATEGORY) + " + sdm_terms),
    ("diff_cat_sdm",  "log_delta ~ fire_dist_poi + edge_poi_dist_mean + C(OVERALL_CATEGORY) + " + sdm_terms),
]

In [ ]:
all_results = []
fitted_models = {}

for model_name, formula in model_specs:
    print(f"Running model: {model_name}")

    model = sm.OLS.from_formula(
        formula=formula,
        data=cbg_agg_census
    ).fit()

    fitted_models[model_name] = model  # optional: keep models

    res_df = extract_model_results(model, model_name)
    all_results.append(res_df)

In [ ]:
results_df = pd.concat(all_results, ignore_index=True)
results_df

In [ ]:
results_edges_all = os.path.join(base, "for_ABM", "reg_edges_all_cbg.csv")
results_df.to_csv(results_edges_all, index=False)

# Category x distance - Aggregation

In [ ]:
revision = "R8.1"
interactions = os.path.join(base, "for_ABM", f"poi_agg_interac_nonevac_{revision}.csv")
poi_agg_census = pd.read_csv(interactions)
poi_agg_census.head()

In [ ]:
# poi_agg_census = (
#     poi_agg_census[["SUB_CATEGORY", "TOP_CATEGORY"]]
#     .drop_duplicates()
# )

In [ ]:
def compute_joint_cat_fire_counts(df, visit_col, phase_label):
    return (
        df.groupby(["TOP_CATEGORY", "fire_distance_bin"], as_index=False)[visit_col]
          .sum()
          .rename(columns={visit_col: f"count_{phase_label}"})
    )

In [ ]:
poi_agg_pre = compute_joint_cat_fire_counts(
    poi_agg_final,
    visit_col="pre_sum",
    phase_label="pre"
)

poi_agg_post = compute_joint_cat_fire_counts(
    poi_agg_final,
    visit_col="post_sum",
    phase_label="post"
)

In [ ]:
poi_joint_agg = (
    poi_agg_pre
    .merge(
        poi_agg_post,
        on=["TOP_CATEGORY", "fire_distance_bin"],
        how="outer"
    )
    .fillna(0)
)

In [ ]:
# poi_joint_agg = poi_joint_agg.merge(
#     subcat_to_overall,
#     on="TOP_CATEGORY",
#     how="left"
# )

In [ ]:
poi_joint_agg["delta"] = np.where(
    poi_joint_agg["count_pre"] > 0,
    (poi_joint_agg["count_post"] - poi_joint_agg["count_pre"]) / poi_joint_agg["count_pre"],
    np.nan
)
# poi_joint_agg["is_new_post"] = (
#     (poi_joint_agg["pre_sum"] == 0) & (poi_joint_agg["post_sum"] > 0)
# )
# poi_joint_agg["lost_in_post"] = (
#     (poi_joint_agg["post_sum"] == 0) & (poi_joint_agg["pre_sum"] > 0)
# )
poi_joint_agg.head()

In [ ]:
vals = poi_joint_agg["count_pre"]

plt.figure(figsize=(7,4))
sns.histplot(vals, bins=100, kde=True, color="purple", alpha=0.7)

plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.title("POI-level change in dyadic connectivity", fontsize=14)
plt.xlabel("(post − pre) / pre  — summed across dyads", fontsize=12)
plt.ylabel("Number of POIs", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import re 

In [ ]:
def parse_interval(x):
    """Convert a string '(a, b]' or '[a, b)' into a pandas.Interval."""
    if isinstance(x, pd.Interval):
        return x
    if isinstance(x, str):
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", x)
        if len(nums) == 2:
            left, right = map(float, nums)
            closed = "right" if x.strip().endswith("]") else "left"
            return pd.Interval(left, right, closed=closed)
    return np.nan  # if totally unparsable


poi_joint_agg["fire_distance_bin"] = poi_joint_agg["fire_distance_bin"].apply(parse_interval)
sorted_bins_f = sorted(
    poi_joint_agg["fire_distance_bin"].dropna().unique(),
    key=lambda x: x.left)

In [ ]:
def interval_label(iv):
    if pd.isna(iv):
        return ""
    return f"{iv.left:.2f} – {iv.right:.2f}"

poi_joint_agg["fire_distance_bin_label"] = poi_joint_agg["fire_distance_bin"].apply(interval_label)
poi_joint_agg.head()

### Heat Map

In [ ]:
# poi_joint_agg["FULL_CATEGORY"] = (
#     poi_joint_agg["TOP_CATEGORY"]
#     + " — "
#     + poi_joint_agg["SUB_CATEGORY"]
# )

In [ ]:
ordered_labels = sorted(
    poi_joint_agg["fire_distance_bin_label"].unique(),
    key=lambda s: float(s.split("–")[0])
)

heat = poi_joint_agg.pivot(
    index="TOP_CATEGORY",
    columns="fire_distance_bin_label",
    values="delta"
)

heat_trimmed = (
    heat.mask(heat > 5.0)
        .reindex(columns=ordered_labels)
)

In [ ]:
import textwrap

def wrap_label(s, width=40):
    return "\n".join(textwrap.wrap(s, width=width))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))

sns.heatmap(
    heat_trimmed,
    cmap="vlag",
    center=0,
    linewidths=0.2,
    linecolor="gray",
    cbar_kws={"label": "ΔP = (P_post - P_pre) / P_pre", "pad": 0.08},
    ax=ax
)

# axis labels and title
ax.set_xlabel("Distance bin")
ax.set_ylabel("POI Category")
ax.set_title("Joint Probability Shift by Category × Distance")

# wrap y-axis labels
ax.set_yticklabels(
    [wrap_label(lbl, width=40) for lbl in heat_trimmed.index],
    fontsize=11
)

# rotate x labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

# category level stats

In [ ]:
revision = "R8.1"
interactions = os.path.join(base, "for_ABM", f"poi_agg_interac_nonevac_{revision}.csv")
poi_agg_census = pd.read_csv(interactions)
poi_agg_census.head()

In [ ]:
cat_agg = (
    poi_agg_census
    .groupby("TOP_CATEGORY", as_index=False)[
        ["Oct2021", "Nov2021", "Jan2022", "Feb2022", "pre_sum", "post_sum"]
    ]
    .sum())
cat_agg.head()

In [ ]:
cat_agg["delta_cat"] = np.where(
    cat_agg["pre_sum"] > 0,
    (cat_agg["post_sum"] - cat_agg["pre_sum"]) / cat_agg["pre_sum"],
    np.nan)
cat_agg["is_new_post"] = ((cat_agg["pre_sum"] == 0) & (cat_agg["post_sum"] > 0))
cat_agg["lost_in_post"] = ((cat_agg["post_sum"] == 0) & (cat_agg["pre_sum"] > 0))
cat_agg.sort_values("delta_cat", ascending=False).head(10)

In [ ]:
cat_delta = (
    cat_agg
    .copy()
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["delta_cat"])
)

cat_delta = cat_delta.sort_values("delta_cat", ascending=False)

In [ ]:
cat_agg["TOP_CATEGORY"].unique()

In [ ]:
import pandas as pd

# --- 1) rename map (raw -> pretty), EXACTLY as coeff plot ---
cat_pretty_map = {
    "Restaurants and Other Eating Places": "Restaurants & Eating Places",
    "Sporting Goods, Hobby, and Musical Instrument Stores": "Sporting/Hobby/Music Stores",
    "Drinking Places (Alcoholic Beverages)": "Bars & Drinking Places",
    "Other Amusement and Recreation Industries": "Amusement & Recreation",
    "Motion Picture and Video Industries": "Film & Video",
    "Museums, Historical Sites, and Similar Institutions": "Museums & Historical Sites",
    "Bakeries and Tortilla Manufacturing": "Bakeries & Tortillas",
    "Book Stores and News Dealers": "Bookstores & Newsstands",
    "Performing Arts Companies": "Performing Arts",
    "Personal Care Services": "Personal Care",
    "Specialty Food Stores": "Specialty Food Stores",
    "Other Miscellaneous Store Retailers": "Other Retail",
    "Amusement Parks and Arcades": "Parks and Arcades",   # ✅ this was missing
}

# --- 2) baseline/reference category (RAW) ---
REF_RAW = "Amusement Parks and Arcades"
REF_PRETTY = cat_pretty_map.get(REF_RAW, REF_RAW)  # ✅ safe even if missing

# --- 3) category order as in the COEFF PLOT (PRETTY NAMES), top -> bottom ---
# Put the reference at the bottom (since it has no dummy coefficient)
cat_order_pretty = [
    "Sporting/Hobby/Music Stores",
    "Specialty Food Stores",
    "Restaurants & Eating Places",
    "Personal Care",
    "Performing Arts",
    "Other Retail",
    "Amusement & Recreation",
    "Museums & Historical Sites",
    "Film & Video",
    "Bars & Drinking Places",
    "Bookstores & Newsstands",
    "Bakeries & Tortillas",
    REF_PRETTY,  # ✅ reference at bottom: "Parks and Arcades"
]

# --- 4) apply pretty names + enforce order ---
cat_delta_plot = cat_delta.copy()

# pretty label (fallback to raw if not in map)
cat_delta_plot["TOP_CATEGORY_pretty"] = cat_delta_plot["TOP_CATEGORY"].map(cat_pretty_map).fillna(cat_delta_plot["TOP_CATEGORY"])

cat_delta_plot["TOP_CATEGORY_pretty"] = pd.Categorical(
    cat_delta_plot["TOP_CATEGORY_pretty"],
    categories=cat_order_pretty,
    ordered=True
)

# Optional sanity check: see if anything didn't map / isn't in order list
missing_in_order = set(cat_delta_plot["TOP_CATEGORY_pretty"].dropna().unique()) - set(cat_order_pretty)
print("Missing from cat_order_pretty:", missing_in_order)

In [ ]:
# 3) Build plotting df from your cat_delta (already computed above)
cat_delta_plot = cat_delta.copy()

# pretty labels
cat_delta_plot["TOP_CATEGORY_pretty"] = cat_delta_plot["TOP_CATEGORY"].map(cat_pretty_map)

# (optional) if anything didn't map, keep original
cat_delta_plot["TOP_CATEGORY_pretty"] = cat_delta_plot["TOP_CATEGORY_pretty"].fillna(cat_delta_plot["TOP_CATEGORY"])

# enforce order
cat_delta_plot["TOP_CATEGORY_pretty"] = pd.Categorical(
    cat_delta_plot["TOP_CATEGORY_pretty"],
    categories=cat_order_pretty,
    ordered=True
)

# drop anything not in the ordering (protects against typos/mismatches)
cat_delta_plot = cat_delta_plot.dropna(subset=["TOP_CATEGORY_pretty"]).copy()

# 4) Plot (grey)
plt.figure(figsize=(6, 5))
sns.barplot(
    data=cat_delta_plot.sort_values("TOP_CATEGORY_pretty"),
    x="delta_cat",
    y="TOP_CATEGORY_pretty",
    color="grey"
)

plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Relative Change in Interaction Weight  (post − pre) / pre")
plt.ylabel("POI Category")
plt.title("")
plt.tight_layout()
plt.show()

In [ ]:
# subcat_agg = (
#     dyad_poi_wide_merged
#     .groupby("SUB_CATEGORY", as_index=False)
#     .agg({
#         "pre_sum": "sum",
#         "post_sum": "sum"
#     })
# )

# subcat_agg["delta_post_minus_pre"] = np.where(
#     subcat_agg["pre_sum"] > 0,
#     (subcat_agg["post_sum"] - subcat_agg["pre_sum"]) / subcat_agg["pre_sum"],
#     np.nan
# )

# subcat_agg = (
#     subcat_agg
#     .replace([np.inf, -np.inf], np.nan)
#     .dropna(subset=["delta_post_minus_pre"])
#     .sort_values("delta_post_minus_pre", ascending=False)
# )

# print("✅ subcat_agg:", subcat_agg.shape)
# subcat_agg.head()

In [ ]:
plt.figure(figsize=(11, 10))

sns.barplot(
    data=subcat_agg,
    x="delta_post_minus_pre",
    y="SUB_CATEGORY",
    color="teal"
)

plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.xlabel("Relative Change in Interaction Weight  (post − pre) / pre")
plt.ylabel("POI Sub-category")
plt.title("Change in POI-Level Social Interaction by Sub-category (Post vs Pre)")

plt.tight_layout()
plt.show()

In [ ]:
cat_plot = cat_agg.copy()

# Total interaction mass per phase
total_pre  = cat_plot["pre_sum"].sum()
total_post = cat_plot["post_sum"].sum()

# Proportions
cat_plot["pre_prop"]  = cat_plot["pre_sum"]  / total_pre
cat_plot["post_prop"] = cat_plot["post_sum"] / total_post

# Delta already exists as relative change
cat_plot["delta"] = cat_plot["delta_cat"]

In [ ]:
ocat_agg = (
    dyad_poi_wide_merged
    .groupby("TOP_CATEGORY", as_index=False)
    .agg({
        "pre_sum": "sum",
        "post_sum": "sum"
    })
)

ocat_agg["delta_post_minus_pre"] = np.where(
    ocat_agg["pre_sum"] > 0,
    (ocat_agg["post_sum"] - ocat_agg["pre_sum"]) / ocat_agg["pre_sum"],
    np.nan
)

ocat_agg = (
    ocat_agg
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["delta_post_minus_pre"])
    .sort_values("delta_post_minus_pre", ascending=False)
)

print("✅ ocat_agg:", ocat_agg.shape)
ocat_agg.head()

In [ ]:
plt.figure(figsize=(4, 4))

sns.barplot(
    data=ocat_agg,
    x="delta_post_minus_pre",
    y="TOP_CATEGORY",
    color="teal"
)

plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.xlabel("Relative Change(post − pre) / pre")
plt.ylabel("POI category")
#plt.title("Change in POI-Level Social Interaction by category (Post vs Pre)")

plt.tight_layout()
plt.show()